# Step 4A.3 — аудит перед фіксацією feature contract (B0–B4)
Не будує жодної моделі. Перевіряє:
1. Точний список і типи 21 колонки `cve_forecast_dataset.parquet`.
2. НАЙВАЖЛИВІШЕ: чи `cvss_score`/`cwe_ids`/`products` могли бути оновлені ПІСЛЯ `t_ref`
   (порівняння з `last_modified` із `cve_master.parquet`, яке не увійшло у forecast-датасет) —
   потенційний темпоральний витік метаданих, окремий від уже перевіреного EPSS/KEV.
3. Наскільки повні CPE/vendor-product дані для побудови додаткових ознак.

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CTI_DIR = Path("/srv/ids_research/data/cti")
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet")
print(f"cve_forecast_dataset: {len(ds):,} рядків, {len(ds.columns)} колонок\n")
print("Точний список колонок і типів:")
for c in ds.columns:
    print(f"  {c:28s} {str(ds[c].dtype):12s} null={ds[c].isna().mean():.1%}")

cve_forecast_dataset: 379,036 рядків, 21 колонок

Точний список колонок і типів:
  cve_id                       str          null=0.0%
  published                    datetime64[us] null=0.0%
  t_ref                        datetime64[us] null=0.0%
  epss_lag_days                int64        null=0.0%
  post_epss_published          bool         null=0.0%
  temporal_consistency         str          null=0.0%
  epss_t                       float64      null=0.0%
  epss_snapshot_date           datetime64[us] null=0.0%
  epss_model_version           str          null=0.0%
  cvss_score                   float64      null=23.0%
  cwe_ids                      object       null=0.0%
  products                     object       null=0.0%
  description                  str          null=0.0%
  future_kev_180d              float64      null=12.9%
  future_kev_365d              float64      null=19.8%
  future_kev_status_180d       str          null=0.0%
  future_kev_status_365d       str          nu

In [2]:
# ---------- НАЙВАЖЛИВІШЕ: аудит потенційного витоку метаданих через last_modified ----------
master = pd.read_parquet(CTI_DIR / "cve_master.parquet", columns=["cve_id", "last_modified"])
master["last_modified"] = pd.to_datetime(master["last_modified"], errors="coerce")

ds2 = ds.merge(master, on="cve_id", how="left")
t_ref = pd.to_datetime(ds2["t_ref"])

# ТРИ окремі, явні категорії замість одного boolean "risk" — NaT (немає last_modified) НЕ
# трактуємо мовчки як "безпечно": ми просто НЕ ЗНАЄМО для таких рядків, чи були пізніші зміни.
metadata_status = np.select(
    [ds2["last_modified"].isna(),
     ds2["last_modified"] > t_ref,
     ds2["last_modified"] <= t_ref],
    ["missing_last_modified", "later_update", "no_later_update"],
    default="?")
ds2["metadata_update_status"] = metadata_status
assert (ds2["metadata_update_status"] != "?").all()

print("Розподіл metadata_update_status (три явні категорії, а не один boolean):")
print(ds2["metadata_update_status"].value_counts())
print(f"  (частки: {ds2['metadata_update_status'].value_counts(normalize=True).to_dict()})")

later = ds2["metadata_update_status"] == "later_update"
print(f"\nCVE з ознакою пізнішого оновлення (last_modified > t_ref): {int(later.sum()):,} ({later.mean():.2%})")

lag = (ds2["last_modified"] - t_ref).dt.days
print("\nРозподіл (last_modified - t_ref) у днях, ЛИШЕ серед 'later_update':")
print(lag[later].describe())

# УНІФІКОВАНО: скрізь у цій комірці — числовий бінарний таргет future_kev_180d, а не
# category future_kev_status_180d, щоб уникнути плутанини між двома різними іменами колонок.
print("\nЧастка кожного metadata_update_status окремо для future_kev_180d==1 і ==0 (H=180):")
print(pd.crosstab(ds2["future_kev_180d"], ds2["metadata_update_status"], normalize="index"))

# ---- ПОСИЛЕНИЙ АУДИТ: completeness метаданих по трьох категоріях ----
# Сам факт "later_update" ще не доводить, що CVSS/CWE/products РЕАЛЬНО змінились — це
# можливість забруднення (potential temporal metadata leakage), не підтверджений факт.
# Порівняння completeness дає непрямий, але конкретний доказ.
def completeness(mask, label):
    sub = ds2[mask]
    has_cvss = sub["cvss_score"].notna().mean()
    has_cwe = sub["cwe_ids"].map(lambda x: x is not None and len(x) > 0).mean()
    has_products = sub["products"].map(lambda x: x is not None and len(x) > 0).mean()
    print(f"{label:38s} n={len(sub):>7,}  CVSS={has_cvss:6.1%}  CWE={has_cwe:6.1%}  products={has_products:6.1%}")

print("\n" + "=" * 70)
print("Completeness метаданих по трьох категоріях metadata_update_status")
print("=" * 70)
for status in ["no_later_update", "later_update", "missing_last_modified"]:
    completeness(ds2["metadata_update_status"] == status, status)

print("\n" + "=" * 70)
print("Та сама completeness, окремо для future_kev_180d==1 і ==0")
print("— критично: чи саме серед майбутніх позитивів 'later_update' супроводжується вищою completeness")
print("=" * 70)
is_pos = ds2["future_kev_180d"] == 1
is_neg = ds2["future_kev_180d"] == 0
for target_mask, target_label in [(is_pos, "future_kev_180d == 1"), (is_neg, "future_kev_180d == 0")]:
    print(f"\n-- {target_label} --")
    for status in ["no_later_update", "later_update", "missing_last_modified"]:
        completeness(target_mask & (ds2["metadata_update_status"] == status), f"  {status}")

print("\nℹ ТЕРМІНОЛОГІЯ ДЛЯ МЕТОДОЛОГІЇ: 'later_update' (last_modified > t_ref) означає лише")
print("  ВІДСУТНІСТЬ ознаки пізнішого оновлення чи ЇЇ наявність — 'potential temporal metadata")
print("  leakage', НЕ підтверджений факт витоку для X% записів. 'missing_last_modified' — це")
print("  третій, окремий стан НЕВІДОМОСТІ, а не автоматично 'безпечно'. Історичного знімка")
print("  NVD-метаданих (на відміну від EPSS) ми не маємо, тому остаточно довести відсутність")
print("  витоку для 'no_later_update' також неможливо — можна лише констатувати відсутність")
print("  цієї конкретної ознаки ризику.")


Розподіл metadata_update_status (три явні категорії, а не один boolean):
metadata_update_status
later_update       348963
no_later_update     30073
Name: count, dtype: int64
  (частки: {'later_update': 0.9206592513639865, 'no_later_update': 0.07934074863601347})

CVE з ознакою пізнішого оновлення (last_modified > t_ref): 348,963 (92.07%)

Розподіл (last_modified - t_ref) у днях, ЛИШЕ серед 'later_update':
count    348963.000000
mean       1139.248631
std         651.332021
min           0.000000
25%         501.000000
50%        1508.000000
75%        1569.000000
max        1983.000000
dtype: float64

Частка кожного metadata_update_status окремо для future_kev_180d==1 і ==0 (H=180):
metadata_update_status  later_update  no_later_update
future_kev_180d                                      
0.0                         0.966158         0.033842
1.0                         1.000000         0.000000

Completeness метаданих по трьох категоріях metadata_update_status
no_later_update          

In [3]:
# ---------- Повнота CWE / CVSS / products для потенційних B2-ознак ----------
has_cwe = ds["cwe_ids"].map(lambda x: x is not None and len(x) > 0)
has_cvss = ds["cvss_score"].notna()
has_products = ds["products"].map(lambda x: x is not None and len(x) > 0)
print(f"З CWE:      {has_cwe.mean():.1%}")
print(f"З CVSS:     {has_cvss.mean():.1%}")
print(f"З products: {has_products.mean():.1%}")

print("\nТоп-20 найпоширеніших CWE (для рішення, скільки multi-hot колонок робити):")
from collections import Counter
cwe_counter = Counter()
for lst in ds["cwe_ids"]:
    if lst is not None:
        cwe_counter.update(lst)
for cwe, cnt in cwe_counter.most_common(20):
    print(f"  {cwe:12s} {cnt:6,} ({cnt/len(ds):.2%})")
print(f"\nВсього унікальних CWE-кодів у датасеті: {len(cwe_counter)}")

З CWE:      81.2%
З CVSS:     77.0%
З products: 82.8%

Топ-20 найпоширеніших CWE (для рішення, скільки multi-hot колонок робити):
  CWE-79       46,527 (12.28%)
  CWE-89       20,448 (5.39%)
  CWE-787      14,641 (3.86%)
  CWE-119      14,187 (3.74%)
  CWE-20       13,070 (3.45%)
  CWE-200      10,823 (2.86%)
  CWE-22        9,991 (2.64%)
  CWE-352       9,607 (2.53%)
  CWE-125       9,423 (2.49%)
  CWE-862       9,412 (2.48%)
  CWE-416       8,291 (2.19%)
  CWE-284       7,423 (1.96%)
  CWE-94        6,951 (1.83%)
  CWE-78        6,512 (1.72%)
  CWE-476       5,595 (1.48%)
  CWE-264       5,493 (1.45%)
  CWE-74        5,212 (1.38%)
  CWE-287       4,736 (1.25%)
  CWE-120       4,393 (1.16%)
  CWE-434       4,336 (1.14%)

Всього унікальних CWE-кодів у датасеті: 799


In [4]:
# ---------- cve_age_days: перевірка (t_ref - published), без сирих дат ----------
ds["cve_age_days"] = (pd.to_datetime(ds["t_ref"]) - pd.to_datetime(ds["published"])).dt.days
print("cve_age_days (=epss_lag_days за визначенням — перевірка, що це те саме поле):")
print((ds["cve_age_days"] == ds["epss_lag_days"]).mean(), "— мають збігатися (обидва = t_ref - published)")
print("\ncve_age_days розподіл:")
print(ds["cve_age_days"].describe())

cve_age_days (=epss_lag_days за визначенням — перевірка, що це те саме поле):
1.0 — мають збігатися (обидва = t_ref - published)

cve_age_days розподіл:
count    379036.000000
mean       1153.446525
std        1930.572435
min           0.000000
25%          14.000000
50%          26.000000
75%        1435.000000
max       12203.000000
Name: cve_age_days, dtype: float64
